In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

CLEANED_PATH = "cleaned_phase1_data.csv"

# خواندن فایل‌ها
udslong = pd.read_csv('alzheimer/udslong_imputed.csv')
mriscan = pd.read_csv('alzheimer/mriscanclean.csv')

# ادغام بر اساس NACCID
combined_df = pd.merge(udslong, mriscan, on='NACCID', how='inner')
print(f"Total merged rows: {len(combined_df)}")


Total merged rows: 1237


In [3]:
# ==========================================================================
# SECTION 1: SETUP, CLEANING & VALIDATION
# ==========================================================================
from pathlib import Path

TABLES = Path("tables")
FIGURES = Path("figures")
TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

BRAIN_COLS = [
    "LH_ENTORHINAL_GVOL", "RH_ENTORHINAL_GVOL",
    "LH_PARAHIPPOCAMPAL_GVOL", "RH_PARAHIPPOCAMPAL_GVOL",
    "LH_PRECUNEUS_GVOL", "RH_PRECUNEUS_GVOL",
    "LH_INSULA_GVOL", "RH_INSULA_GVOL",
]
KEY_VARS = (
    ["NACCAGE", "SEX", "EDUC", "INDEPEND", "MOCATOTS", "CDRGLOB", "NACCUDSD",
     "DIABETES", "HYPERTEN", "CBSTROKE"] + BRAIN_COLS
)

if "NACCUDSD" not in combined_df.columns and "NACCUDSD_x" in combined_df.columns:
    combined_df = (combined_df.drop(columns=["NACCUDSD_y"])
                   .rename(columns={"NACCUDSD_x": "NACCUDSD"}))

clean_df = combined_df.copy()
cleaning_log = []

def log_step(variable, rule, n, action, reason):
    cleaning_log.append({"variable": variable, "rule": rule, "n_affected": n,
                         "action": action, "reason": reason})
    print(f"  [{variable}] {rule}: {n} -> {action}")

def set_nan(df, col, mask, rule, reason):
    n = int(mask.sum())
    df.loc[mask, col] = np.nan
    log_step(col, rule, n, "set to NaN", reason)

print("--- 3 & 4. MoCA Score Validation & Cleaning ---")
set_nan(clean_df, "MOCATOTS", clean_df["MOCATOTS"] == 88, "value == 88", "special code")
set_nan(clean_df, "MOCATOTS", clean_df["MOCATOTS"] > 30, "value > 30", "outside valid range")
clean_df["MOCA_ZERO_FLAG"] = clean_df["MOCATOTS"] == 0
log_step("MOCATOTS", "value == 0", int(clean_df["MOCA_ZERO_FLAG"].sum()), "kept + flagged", "extreme value")

valid_moca = clean_df["MOCATOTS"].notna()
clean_df["MOCA_NONINT_FLAG"] = valid_moca & (clean_df["MOCATOTS"] % 1 != 0)
log_step("MOCATOTS", "non-integer value", int(clean_df["MOCA_NONINT_FLAG"].sum()), "kept + flagged", "likely imputed")

SPECIAL_CODES = {"EDUC": [99], "INDEPEND": [9], "HYPERTEN": [9], "DIABETES": [9], "CBSTROKE": [9]}
for col, codes in SPECIAL_CODES.items():
    set_nan(clean_df, col, clean_df[col].isin(codes), f"value in {codes}", "special code")

for col in BRAIN_COLS:
    set_nan(clean_df, col, clean_df[col] <= 0, "volume <= 0", "physically impossible")

def to_binary_history(s, threshold=0.5):
    return (s >= threshold).astype(float).where(s.notna())

BINARY_VARS = {"HYPERTEN": "HYPERTEN_BIN", "DIABETES": "DIABETES_BIN", "CBSTROKE": "STROKE_BIN"}
for col, new_col in BINARY_VARS.items():
    clean_df[new_col] = to_binary_history(clean_df[col])

analysis_vars = [c for c in KEY_VARS if c in clean_df.columns]
summary_df = pd.DataFrame({
    "Valid N": clean_df[analysis_vars].notna().sum(),
    "Missing N": len(clean_df) - clean_df[analysis_vars].notna().sum(),
    "Missing %": (len(clean_df) - clean_df[analysis_vars].notna().sum()) / len(clean_df) * 100,
})
print("\n--- Missingness Summary ---\n", summary_df.round(2))
summary_df.round(2).to_csv(TABLES / "key_variables_missingness.csv")

n_merged = len(clean_df)
analysis_df = clean_df[clean_df["MOCATOTS"].notna()].copy()
print(f"\nFinal analytic sample: {len(analysis_df)} participants")


--- 3 & 4. MoCA Score Validation & Cleaning ---
  [MOCATOTS] value == 88: 4 -> set to NaN
  [MOCATOTS] value > 30: 5 -> set to NaN
  [MOCATOTS] value == 0: 1 -> kept + flagged
  [MOCATOTS] non-integer value: 244 -> kept + flagged
  [EDUC] value in [99]: 2 -> set to NaN
  [INDEPEND] value in [9]: 2 -> set to NaN
  [HYPERTEN] value in [9]: 11 -> set to NaN
  [DIABETES] value in [9]: 8 -> set to NaN
  [CBSTROKE] value in [9]: 3 -> set to NaN
  [LH_ENTORHINAL_GVOL] volume <= 0: 1 -> set to NaN
  [RH_ENTORHINAL_GVOL] volume <= 0: 0 -> set to NaN
  [LH_PARAHIPPOCAMPAL_GVOL] volume <= 0: 0 -> set to NaN
  [RH_PARAHIPPOCAMPAL_GVOL] volume <= 0: 0 -> set to NaN
  [LH_PRECUNEUS_GVOL] volume <= 0: 0 -> set to NaN
  [RH_PRECUNEUS_GVOL] volume <= 0: 0 -> set to NaN
  [LH_INSULA_GVOL] volume <= 0: 0 -> set to NaN
  [RH_INSULA_GVOL] volume <= 0: 0 -> set to NaN

--- Missingness Summary ---
                          Valid N  Missing N  Missing %
NACCAGE                     1237          0       0.00
S

In [4]:
# ==========================================================================
# SECTION 2: VASCULAR HISTORY & OUTLIER REPORTING (NO REMOVAL)
# ==========================================================================
def moca_by_binary(df, col, label_no, label_yes, title, fname):
    d = df[["MOCATOTS", col]].dropna().copy()
    labels = {0.0: label_no, 1.0: label_yes}
    d["group"] = d[col].map(labels)
    g = d.groupby("group")["MOCATOTS"]
    stats = g.agg(n="count", mean="mean", sd="std", median="median",
                  q1=lambda x: x.quantile(0.25), q3=lambda x: x.quantile(0.75))
    stats["iqr"] = stats["q3"] - stats["q1"]
    stats = stats.reindex([label_no, label_yes])
    print(f"\nMoCA by {title}:\n", stats.round(2))
    stats.round(2).to_csv(TABLES / f"{Path(fname).stem}_stats.csv")

    counts = d["group"].value_counts()
    plt.figure(figsize=(7, 5))
    sns.boxplot(data=d, x="group", y="MOCATOTS", order=[label_no, label_yes], hue="group", palette="Set2", legend=False)
    plt.xticks(range(2), [f"{label_no}\n(n={counts.get(label_no,0)})", f"{label_yes}\n(n={counts.get(label_yes,0)})"])
    plt.title(f"MoCA score by {title}")
    plt.savefig(FIGURES / fname, dpi=300, bbox_inches="tight")
    plt.close()

print("\n--- Vascular history vs MoCA ---")
moca_by_binary(analysis_df, "HYPERTEN_BIN", "No hypertension", "Hypertension history", "hypertension history", "bp_vs_moca_boxplot.png")
moca_by_binary(analysis_df, "DIABETES_BIN", "No diabetes", "Diabetes history", "diabetes history", "diabetes_vs_moca_boxplot.png")
moca_by_binary(analysis_df, "STROKE_BIN", "No stroke", "Stroke history", "stroke history", "stroke_vs_moca_boxplot.png")

print("\n--- MRI Outlier Report (IQR rule, nothing removed) ---")
report_rows, flagged_parts = [], []
for col in BRAIN_COLS:
    s = analysis_df[col].dropna()
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    low, high = analysis_df[col] < lo, analysis_df[col] > hi
    report_rows.append({"variable": col, "n_valid": len(s), "min": s.min(), "max": s.max(), "Q1": q1, "Q3": q3, "IQR": iqr, "lower": lo, "upper": hi, "n_low": int(low.sum()), "n_high": int(high.sum())})
    analysis_df[f"OUT_{col}"] = low | high
    flagged = analysis_df.loc[low | high, ["NACCID", col]].rename(columns={col: "value"})
    flagged["variable"], flagged["direction"] = col, np.where(flagged["value"] < lo, "low", "high")
    flagged_parts.append(flagged)

outlier_report = pd.DataFrame(report_rows)
outlier_ids = pd.concat(flagged_parts, ignore_index=True)
print(outlier_report.round(1).to_string(index=False))
outlier_report.to_csv(TABLES / "mri_outlier_report.csv", index=False)
outlier_ids.to_csv(TABLES / "mri_outlier_ids.csv", index=False)
analysis_df["MRI_OUTLIER_COUNT"] = analysis_df[[f"OUT_{c}" for c in BRAIN_COLS]].sum(axis=1)



--- Vascular history vs MoCA ---

MoCA by hypertension history:
                         n   mean    sd  median    q1    q3  iqr
group                                                          
No hypertension       738  24.77  4.40    26.0  23.0  27.6  4.6
Hypertension history  479  23.87  4.35    25.0  22.0  27.0  5.0

MoCA by diabetes history:
                      n   mean    sd  median    q1    q3  iqr
group                                                       
No diabetes       1093  24.58  4.37    25.8  23.0  27.0  4.0
Diabetes history   127  23.10  4.62    24.2  21.0  26.0  5.0

MoCA by stroke history:
                    n   mean    sd  median    q1    q3  iqr
group                                                     
No stroke       1208  24.41  4.42    25.4  23.0  27.0  4.0
Stroke history    17  24.36  4.41    25.0  22.0  28.0  6.0

--- MRI Outlier Report (IQR rule, nothing removed) ---
               variable  n_valid   min     max     Q1      Q3    IQR  lower   upper  n_l

In [5]:
# ==========================================================================
# SECTION 3: CORRELATION & FINAL EXPORT
# ==========================================================================
print("\n" + "=" * 50 + "\n--- BRAIN CORRELATIONS ---\n" + "=" * 50)
regions_to_analyze = {
    "ENTORHINAL": ['LH_ENTORHINAL_GVOL', 'RH_ENTORHINAL_GVOL'],
    "PARAHIPPOCAMPAL": ['LH_PARAHIPPOCAMPAL_GVOL', 'RH_PARAHIPPOCAMPAL_GVOL'],
    "PRECUNEUS": ['LH_PRECUNEUS_GVOL', 'RH_PRECUNEUS_GVOL'],
    "INSULA": ['LH_INSULA_GVOL', 'RH_INSULA_GVOL']
}

for group_name, columns in regions_to_analyze.items():
    for col in columns:
        valid_data = analysis_df[['MOCATOTS', col]].dropna()
        if not valid_data.empty:
            r_val = valid_data[col].corr(valid_data['MOCATOTS'])
            print(f"{col:<25} | Pearson R: {r_val:.4f}")
            plt.figure(figsize=(7, 5))
            sns.regplot(x=col, y='MOCATOTS', data=valid_data, scatter_kws={'alpha': 0.5}, line_kws={'color': 'red'})
            plt.title(f'{col} vs MoCA Score')
            plt.savefig(FIGURES / f"{col.lower()}_vs_moca_scatter.png", dpi=300)
            plt.close()

# Final Clean Save
pd.DataFrame(cleaning_log).to_csv(TABLES / "cleaning_log.csv", index=False)
analysis_df.to_csv("cleaned_phase1_data.csv", index=False)
print(f"\n[DONE] All files saved in 'tables/' and 'figures/'.")
print(f"[DONE] Final dataset saved as 'cleaned_phase1_data.csv'")



--- BRAIN CORRELATIONS ---
LH_ENTORHINAL_GVOL        | Pearson R: 0.2119
RH_ENTORHINAL_GVOL        | Pearson R: 0.2070
LH_PARAHIPPOCAMPAL_GVOL   | Pearson R: 0.2152
RH_PARAHIPPOCAMPAL_GVOL   | Pearson R: 0.2396
LH_PRECUNEUS_GVOL         | Pearson R: 0.2564
RH_PRECUNEUS_GVOL         | Pearson R: 0.2292
LH_INSULA_GVOL            | Pearson R: 0.1303
RH_INSULA_GVOL            | Pearson R: 0.1371

[DONE] All files saved in 'tables/' and 'figures/'.
[DONE] Final dataset saved as 'cleaned_phase1_data.csv'


In [ ]:
import os
import pandas as pd

if "combined_df" not in globals():
    if os.path.exists("cleaned_phase1_data.csv"):
        combined_df = pd.read_csv("cleaned_phase1_data.csv")
    elif os.path.exists("tables/cleaned_phase1_data.csv"):
        combined_df = pd.read_csv("tables/cleaned_phase1_data.csv")
    else:
        # اگر فایل نبود، از مسیر اصلی می‌خوانیم
        combined_df = pd.read_csv("alz/alzheimer/udslong_imputed.csv")
    print(f"[OK] Data loaded. Shape: {combined_df.shape}")
else:
    print(f"[OK] combined_df is already in memory. Shape: {combined_df.shape}")


[OK] combined_df is already in memory. Shape: (1228, 318)


In [7]:
# Block 0.5: Verify which file is loaded
import os
print("Loaded file:", getattr(combined_df, "attrs", {}).get("source", "unknown"))
print("Shape:", combined_df.shape)
print("MoCA zeros:", (combined_df["MOCATOTS"] == 0).sum())
print("MoCA < 10:", (combined_df["MOCATOTS"] < 10).sum())
print("MoCA describe:\n", combined_df["MOCATOTS"].describe())


Loaded file: unknown
Shape: (1228, 318)
MoCA zeros: 1
MoCA < 10: 20
MoCA describe:
 count    1228.000000
mean       24.392997
std         4.424823
min         0.000000
25%        23.000000
50%        25.400000
75%        27.000000
max        30.000000
Name: MOCATOTS, dtype: float64


In [ ]:
# Frequency and Percentage for Categorical Variables
cat_vars = ["SEX", "INDEPEND", "HYPERTEN", "DIABETES", "CBSTROKE"]

for var in cat_vars:
    if var in combined_df.columns:
        freq = combined_df[var].value_counts(dropna=False).sort_index()
        pct = (freq / len(combined_df) * 100).round(1)
        
        df_cat = freq.to_frame("Frequency (N)").assign(Percent=pct)
        print(f"\n--- Frequency Table for: {var} ---")
        print(df_cat)
    else:
        print(f"[Warning] Variable '{var}' not found in DataFrame.")



--- Frequency Table for: SEX ---
     Frequency (N)  Percent
SEX                        
1.0            496     40.4
2.0            732     59.6

--- Frequency Table for: INDEPEND ---
          Frequency (N)  Percent
INDEPEND                        
1.0                1116     90.9
2.0                  83      6.8
3.0                  26      2.1
4.0                   1      0.1
NaN                   2      0.2

--- Frequency Table for: HYPERTEN ---
          Frequency (N)  Percent
HYPERTEN                        
0.0                 726     59.1
0.2                   7      0.6
0.4                   5      0.4
0.6                   7      0.6
1.0                 442     36.0
1.8                   1      0.1
2.0                  28      2.3
2.4                   1      0.1
NaN                  11      0.9

--- Frequency Table for: DIABETES ---
          Frequency (N)  Percent
DIABETES                        
0.0                1084     88.3
0.2                   7      0.6
0.4        

In [6]:
import numpy as np

# Block 2: Summary Statistics for Numerical Variables
num_vars = ["NACCAGE", "MOCATOTS", "LH_ENTORHINAL_GVOL", "RH_ENTORHINAL_GVOL", 
            "LH_PRECUNEUS_GVOL", "RH_PRECUNEUS_GVOL"]

summary_list = []

for var in num_vars:
    if var in combined_df.columns:
        s = combined_df[var].dropna()
        summary_list.append({
            "Variable": var,
            "Mean": round(s.mean(), 2),
            "Std": round(s.std(), 2),
            "Median": round(s.median(), 2),
            "IQR": round(s.quantile(0.75) - s.quantile(0.25), 2),
            "Min": round(s.min(), 2),
            "Max": round(s.max(), 2),
            "Missing N": combined_df[var].isna().sum()
        })

df_summary = pd.DataFrame(summary_list).set_index("Variable")
print("\n--- Summary Statistics for Numerical Variables ---")
print(df_summary)



--- Summary Statistics for Numerical Variables ---
                       Mean      Std  Median      IQR    Min      Max  \
Variable                                                                
NACCAGE               68.18     8.06    68.0    11.00   41.0     91.0   
MOCATOTS              24.39     4.42    25.4     4.00    0.0     30.0   
LH_ENTORHINAL_GVOL  1556.67   411.74  1545.0   493.00  211.0   4265.0   
RH_ENTORHINAL_GVOL  1526.25   405.23  1513.5   456.50    6.0   5334.0   
LH_PRECUNEUS_GVOL   8494.67  1416.94  8529.0  1639.75  124.0  12914.0   
RH_PRECUNEUS_GVOL   9266.54  1541.99  9228.0  1815.75  716.0  14351.0   

                    Missing N  
Variable                       
NACCAGE                     0  
MOCATOTS                    0  
LH_ENTORHINAL_GVOL          1  
RH_ENTORHINAL_GVOL          0  
LH_PRECUNEUS_GVOL           0  
RH_PRECUNEUS_GVOL           0  


In [8]:
# Block 5.1: Fix HYPERTEN -> HYPERTEN_BIN
import numpy as np
import pandas as pd

# show raw values first
print("Raw value counts:")
print(combined_df["HYPERTEN"].value_counts(dropna=False).sort_index())

# 9 = missing code in NACC; everything else > 0 means "history of hypertension"
raw = combined_df["HYPERTEN"].replace(9, np.nan)

combined_df["HYPERTEN_BIN"] = np.where(raw.isna(), np.nan,
                                np.where(raw > 0, 1.0, 0.0))

print("\nAfter cleaning:")
print(combined_df["HYPERTEN_BIN"].value_counts(dropna=False).sort_index())
print("Missing:", combined_df["HYPERTEN_BIN"].isna().sum())


Raw value counts:
HYPERTEN
0.0    726
0.2      7
0.4      5
0.6      7
1.0    442
1.8      1
2.0     28
2.4      1
NaN     11
Name: count, dtype: int64

After cleaning:
HYPERTEN_BIN
0.0    726
1.0    491
NaN     11
Name: count, dtype: int64
Missing: 11


In [9]:
# Block 5.2: MoCA by corrected hypertension groups
g = combined_df.groupby("HYPERTEN_BIN")["MOCATOTS"]

summary = pd.DataFrame({
    "n":      g.count(),
    "mean":   g.mean().round(2),
    "sd":     g.std().round(2),
    "median": g.median().round(1),
    "q1":     g.quantile(0.25).round(1),
    "q3":     g.quantile(0.75).round(1),
})
summary["iqr"] = (summary["q3"] - summary["q1"]).round(1)

summary.index = summary.index.map({0.0: "No hypertension", 1.0: "Hypertension history"})
print(summary)


                        n   mean    sd  median    q1    q3  iqr
HYPERTEN_BIN                                                   
No hypertension       726  24.76  4.42    26.0  23.0  27.6  4.6
Hypertension history  491  23.91  4.33    25.0  22.0  27.0  5.0


In [10]:
# Verify participant structure and merge integrity (task J2 & J3)
import pandas as pd

uds = pd.read_csv("alzheimer/udslong_imputed.csv")
mri = pd.read_csv("alzheimer/mriscanclean.csv")

print("UDS rows:", len(uds), "| unique NACCID:", uds["NACCID"].nunique())
print("MRI rows:", len(mri), "| unique NACCID:", mri["NACCID"].nunique())

# J2: check for repeated participants
print("Duplicated NACCID in UDS:", uds["NACCID"].duplicated().sum())
print("Duplicated NACCID in MRI:", mri["NACCID"].duplicated().sum())

# J3: data flow of the merge
merged = uds.merge(mri, on="NACCID", how="inner")
print("Inner merge result:", merged.shape)
print("UDS without MRI match:", len(set(uds["NACCID"]) - set(mri["NACCID"])))
print("MRI without UDS match:", len(set(mri["NACCID"]) - set(uds["NACCID"])))


UDS rows: 1237 | unique NACCID: 1237
MRI rows: 1237 | unique NACCID: 1237
Duplicated NACCID in UDS: 0
Duplicated NACCID in MRI: 0
Inner merge result: (1237, 305)
UDS without MRI match: 0
MRI without UDS match: 0
